# Faruq-v3 ACMC — paired optimization confirmation

Konfirmasi validation-only yang adil. Seed 42 memakai kontrol D0/D0FT/ACMC1 yang sudah PASS. Untuk seed 123 dan 2026 notebook ini membuat tiga jalur dari awal: D0, D0FT dari D0 yang sama, dan ACMC1 dari D0 yang sama. Hasil konfirmasi ACMC lama tidak dipakai. Test tidak tersedia maupun diakses. Semua checkpoint berada di satu folder proyek Google Drive dan dapat dilanjutkan setelah runtime putus.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os, shutil, subprocess, sys, tarfile, time
from pathlib import Path

REPO = Path('/content/coffee-bean-detection')
BRANCH = 'agent/add-vadcp-pipeline'
os.chdir('/content')
if REPO.exists():
    shutil.rmtree(REPO)
clone = ['git', 'clone', '--depth', '1', '--branch', BRANCH, 'https://github.com/ediprin/coffee-bean-detection.git', str(REPO)]
for attempt in range(1, 4):
    result = subprocess.run(clone)
    if result.returncode == 0:
        break
    if REPO.exists():
        shutil.rmtree(REPO)
    if attempt == 3:
        raise RuntimeError('Git clone gagal tiga kali.')
    time.sleep(2)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO)], check=True)
for module_name in list(sys.modules):
    if module_name == 'coffee_detector' or module_name.startswith('coffee_detector.'):
        sys.modules.pop(module_name, None)
sys.path.insert(0, str(REPO / 'src'))
os.chdir(REPO)
print('REPO:', REPO)

In [ ]:
import torch
from coffee_detector.drive_project import require_project_artifact, resolve_drive_project_root

assert torch.cuda.is_available(), 'Aktifkan T4 GPU.'
PROJECT_ROOT = resolve_drive_project_root(required_relative_paths=(
    'bundles/faruq-development-v3-grouped.tar',
    'experiments/faruq-v3-acmc-optimization-control-v1/val_reports/acmc1_optimization_control_seed42.json',
))
ARCHIVE = require_project_artifact(PROJECT_ROOT, 'bundles/faruq-development-v3-grouped.tar')
SEED42_CONTROL = require_project_artifact(PROJECT_ROOT, 'experiments/faruq-v3-acmc-optimization-control-v1/val_reports/acmc1_optimization_control_seed42.json')
DATA_ROOT = Path('/content/faruq-development-v3-grouped')
GROUPED_SUMMARY = DATA_ROOT / 'faruq_grouped_summary.json'
OUTPUT_ROOT = PROJECT_ROOT / 'experiments/faruq-v3-acmc-paired-confirmation-v1'
if not GROUPED_SUMMARY.is_file():
    with tarfile.open(ARCHIVE, 'r') as archive:
        archive.extractall('/content', filter='data')
assert GROUPED_SUMMARY.is_file(), GROUPED_SUMMARY
assert not (DATA_ROOT / 'test').exists(), 'Test tidak boleh tersedia.'
for seed in (123, 2026):
    d0 = OUTPUT_ROOT / 'D0_base' / f'D0_seed{seed}/weights/best.pt'
    d0ft = OUTPUT_ROOT / 'D0FT' / f'D0FT_seed{seed}/weights/best.pt'
    acmc = OUTPUT_ROOT / 'ACMC1' / f'ACMC1_seed{seed}/weights/best.pt'
    print(f'SEED {seed}: D0=' + ('COMPLETE' if d0.is_file() else 'START') + ', D0FT=' + ('COMPLETE' if d0ft.is_file() else 'START') + ', ACMC1=' + ('COMPLETE' if acmc.is_file() else 'START'))
print('GPU    :', torch.cuda.get_device_name(0))
print('PROJECT:', PROJECT_ROOT)
print('OUTPUT :', OUTPUT_ROOT)

In [ ]:
command = [
    sys.executable, '-u', '-m', 'coffee_detector.experiments.run_faruq_v3_acmc_paired_confirmation',
    '--data-root', str(DATA_ROOT),
    '--grouped-summary', str(GROUPED_SUMMARY),
    '--seed42-control', str(SEED42_CONTROL),
    '--output-root', str(OUTPUT_ROOT),
    '--seeds', '123', '2026', '--device', '0', '--authorize-training',
]
print('MENJALANKAN:', ' '.join(command), flush=True)
process = subprocess.Popen(command, cwd=REPO, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1)
for line in process.stdout:
    print(line, end='', flush=True)
return_code = process.wait()
if return_code != 0:
    raise RuntimeError(f'Konfirmasi ACMC gagal, return code={return_code}; traceback lengkap tercetak di atas.')

In [ ]:
import json, pandas as pd
from IPython.display import display

SUMMARY = OUTPUT_ROOT / 'val_reports/acmc1_paired_optimization_confirmation.json'
assert SUMMARY.is_file(), f'Konfirmasi belum selesai: {SUMMARY}'
result = json.loads(SUMMARY.read_text(encoding='utf-8'))
assert result['evaluation_split'] == 'val'
assert result['test_images_accessed'] is False
rows = [{'metric': metric, **values} for metric, values in result['aggregate'].items()]
formatters = {name: '{:.2%}' for name in ('d0_mean', 'd0ft_mean', 'acmc1_mean', 'head_delta_mean', 'head_delta_min')}
display(pd.DataFrame(rows).style.format(formatters))
print('CRITERIA:', result['criteria'])
print('DECISION:', result['decision'])
print('NEXT    :', result['next_action'])
print('SUMMARY :', SUMMARY)
print('Kirim tabel dan keputusan. Jangan membuka test secara manual.')